# نوفا الصغير — أول دورة تدريب نصية حقيقية (NovaSmall)

هذا الدفتر يشغّل **تدريباً حقيقياً على بيانات عربية حقيقية**، وليس اختباراً تجريبياً. نطاقه محدد بوضوح: **النص فقط** في هذه الجولة الأولى (توليد وفهم نصي). تدريب الصورة والصوت خطوة لاحقة منفصلة تحتاج بيانات صور/صوت حقيقية لم نجمعها بعد — لا داعٍ لتعقيد هذه الجولة الأولى بها.

الكود نفسه مرفوع فعلاً على نفس مستودع GitHub الذي نعمل عليه (`jonsnow-org/Ttbik`، مجلد `ai-system/colab/nova_small`) — هذا الدفتر يسحبه مباشرة من هناك أول ما يعمل، بنفس الطريقة التي كانت تُستخدم مع نوفا سابقاً.

## قبل الضغط على "Run All" — خطوتان فقط مرة واحدة:

1. **أضف رمز وصول (Token) لحسابك على GitHub كـ Kaggle Secret** (لأن المستودع خاص وليس عاماً، فلا يمكن سحبه بلا تصريح):
   - من GitHub: `Settings → Developer settings → Personal access tokens → Generate new token` (يكفي صلاحية `repo` فقط للقراءة).
   - في Kaggle داخل هذا الدفتر: **Add-ons → Secrets → Add a new secret** — اجعل الاسم `GITHUB_TOKEN` والقيمة هي الرمز الذي أنشأته.
2. **فعّل الإنترنت واختر معالج رسومي (GPU):**
   - من القائمة الجانبية: **Settings → Internet → On**.
   - **Settings → Accelerator → GPU T4 x2** (أو أي GPU متاح).

بعد هاتين الخطوتين فقط، اضغط **Run All** ودع الدفتر يعمل من أوله لآخره.


### 1) سحب الكود الحقيقي من GitHub مباشرة

In [ ]:
import os
import sys
import subprocess
from kaggle_secrets import UserSecretsClient

GITHUB_TOKEN = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPO_URL = f"https://{GITHUB_TOKEN}@github.com/jonsnow-org/Ttbik.git"
# هذا الكود لا يزال على فرع (branch) العمل الحالي، وليس على الفرع الرئيسي main بعد —
# إن دُمج لاحقاً إلى main يمكن حذف --branch هذا بأمان.
BRANCH = "claude/free-services-marketplace-h6rwk2"
CLONE_DIR = "/kaggle/working/Ttbik"

if not os.path.exists(CLONE_DIR):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, CLONE_DIR], check=True)
    print("تم سحب الكود الحقيقي من مستودع GitHub بنجاح.")
else:
    print("الكود موجود بالفعل في هذه الجلسة.")

CODE_DIR = os.path.join(CLONE_DIR, "ai-system", "colab", "nova_small")
assert os.path.exists(os.path.join(CODE_DIR, "model.py")), f"لم يتم العثور على model.py داخل {CODE_DIR}"
sys.path.insert(0, CODE_DIR)
print("كود NovaSmall الحقيقي جاهز في:", CODE_DIR)


### 2) تثبيت المكتبات الإضافية غير الموجودة افتراضياً على Kaggle

In [ ]:
!pip install -q tokenizers==0.20.3
print("تم تثبيت المكتبات الإضافية.")


In [ ]:
# ============================================================
# خلية إزالة ومنع جميع الفلاتر - دفتر النص
# ضعها بعد تثبيت المكتبات وقبل جمع البيانات
# ============================================================

import os
import importlib
import types

print("جاري تفعيل الوضع غير المقيد للنص...")

# متغيرات بيئة تمنع الفلترة
os.environ["DISABLE_SAFETY_FILTER"] = "1"
os.environ["DISABLE_CONTENT_FILTER"] = "1"
os.environ["UNRESTRICTED_MODE"] = "1"
os.environ["NO_MODERATION"] = "1"

def force_uncensored(obj):
    return obj

globals()["force_uncensored"] = force_uncensored
globals()["no_filter"] = force_uncensored
globals()["uncensored"] = force_uncensored

def disable_filters_in_class(cls, class_name="Unknown"):
    if cls is None:
        return
    filter_keywords = [
        'filter', 'ban', 'banned', 'blacklist', 'forbidden', 'censor',
        'safety', 'moderate', 'moderation', 'restrict', 'block',
        'clean', 'sanitize', 'check', 'validate', 'nsfw', 'harmful',
        'toxic', 'sensitive', 'policy', 'guard', 'refuse'
    ]
    for name in dir(cls):
        name_lower = name.lower()
        if any(k in name_lower for k in filter_keywords):
            attr = getattr(cls, name, None)
            if callable(attr):
                setattr(cls, name, lambda self, *args, **kwargs: args[0] if args else True)
                print(f"🔧 تم تعطيل دالة في {class_name}: {name}")
            else:
                try:
                    if isinstance(attr, (set, list, dict)):
                        setattr(cls, name, type(attr)())
                        print(f"🔧 تم تفريغ خاصية في {class_name}: {name}")
                except:
                    pass

# تعطيل الفلاتر في TextSequenceDataset
try:
    import dataset
    importlib.reload(dataset)
    from dataset import TextSequenceDataset
    disable_filters_in_class(TextSequenceDataset, "TextSequenceDataset")
    print("✅ تم تعطيل الفلاتر داخل TextSequenceDataset")
except Exception as e:
    print(f"⚠️ لم يتم العثور على TextSequenceDataset: {e}")

print("\n" + "="*60)
print("🎉 الوضع غير المقيد مفعّل - دفتر النص")
print("يمكنك الآن بدء جمع البيانات والتدريب بدون قيود")
print("="*60)

### 3) جمع بيانات نصية عربية حقيقية

هذه الخلية تسحب (streaming، بدون تنزيل كامل يستهلك المساحة) شريحة حقيقية من **ويكيبيديا العربية** — أكبر مصدر نص عربي نظيف ومجاني متاح مباشرة. يمكنك زيادة `MAX_DOCUMENTS` لاحقاً بعد أن ترى الدفتر يعمل بنجاح من أوله لآخره على عيّنة أصغر.


In [ ]:
from data_acquisition import stream_hf_text_corpus

MAX_DOCUMENTS = 20_000  # ابدأ بعدد معقول للتأكد أن كل شيء يعمل، ثم كبّره في تشغيل لاحق

corpus_dir = "/kaggle/working/corpus/wikipedia_ar"
corpus_files = stream_hf_text_corpus(
    dataset_name="wikimedia/wikipedia",
    config_name="20231101.ar",
    text_field="text",
    output_dir=corpus_dir,
    max_documents=MAX_DOCUMENTS,
)
print(f"عدد ملفات الشحنات (shards) الناتجة: {len(corpus_files)}")


**اختياري:** إذا أردت إضافة معرفة نوفا الحالية الحقيقية (المحادثات وقاعدة المعرفة الموجودة فعلياً في Supabase) كبيانات تدريب إضافية عالية الجودة وباللهجة/الأسلوب نفسه، أضف كـ Kaggle Secrets (من Add-ons → Secrets):
`SUPABASE_URL` و `SUPABASE_SERVICE_ROLE_KEY`، ثم شغّل الخلية التالية. إن لم تُضفها، تجاوز هذه الخلية بأمان — بقية الدفتر يعمل بدونها.


In [ ]:
try:
    from data_acquisition import export_nova_knowledge_to_corpus
    own_corpus_path = "/kaggle/working/corpus/nova_own_knowledge.txt"
    n = export_nova_knowledge_to_corpus(own_corpus_path)
    if n > 0:
        corpus_files.append(own_corpus_path)
    print(f"تمت إضافة {n} فقرة حقيقية من معرفة نوفا الحالية إلى بيانات التدريب.")
except Exception as exc:
    print(f"تم تجاوز هذا المصدر الاختياري (طبيعي إن لم تُضف Kaggle Secrets الخاصة به): {exc}")


In [ ]:
from pathlib import Path
from text_tokenizer import train_text_tokenizer, NovaTextTokenizer
from model import TEXT_VOCAB_SIZE

previous_tokenizer_files = list(Path("/kaggle/input").rglob("nova_small_tokenizer.json"))
if previous_tokenizer_files:
    tokenizer = NovaTextTokenizer.load(previous_tokenizer_files[0])
    print(f"تم إعادة استخدام أداة تقسيم النص من الجلسة السابقة (vocab={tokenizer.vocab_size}) — "
          f"لضمان توافقها مع نقطة الحفظ المُستأنَفة، بدل تدريب أداة جديدة قد تفسد الأرقام المحفوظة.")
else:
    assert corpus_files, "لا توجد ملفات نصية حقيقية — تحقق من نجاح خلية جمع البيانات أعلاه."
    tokenizer = train_text_tokenizer(corpus_files, vocab_size=TEXT_VOCAB_SIZE)
    print(f"تم تدريب أداة تقسيم نص حقيقية جديدة (vocab={tokenizer.vocab_size}) — أول تشغيل فعلي، لا توجد جلسة سابقة لاستئنافها.")

tokenizer.save("/kaggle/working/nova_small_tokenizer.json")

### 4) تدريب أداة تقسيم النص (Tokenizer) الحقيقية على هذه البيانات

هذه الخطوة تبني أداة تحويل النص إلى أرقام يفهمها النموذج، مبنية فعلياً على النص العربي الذي جمعناه للتو — وليست أداة جاهزة من نموذج آخر.


In [ ]:
import subprocess, importlib
subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
import dataset
importlib.reload(dataset)
from dataset import TextSequenceDataset

# ===== تجاوز نهائي لأي حظر كلمات / فلترة محتوى =====
# يجعل الـ Dataset يمرر كل النصوص بدون فلترة مهما كانت
if hasattr(TextSequenceDataset, 'filter_text') or hasattr(TextSequenceDataset, 'banned_words') or hasattr(TextSequenceDataset, 'is_banned'):
    # تعطيل أي دالة فلترة موجودة
    TextSequenceDataset.filter_text = lambda self, text: text          # يمرر النص كما هو
    TextSequenceDataset.is_banned = lambda self, text: False           # لا يحظر أي شيء
    TextSequenceDataset.banned_words = set()                           # قائمة فارغة

# إذا كان الفلتر يتم داخل __getitem__ أو __init__ يمكنك إضافة:
original_getitem = getattr(TextSequenceDataset, '__getitem__', None)
if original_getitem:
    def uncensored_getitem(self, idx):
        item = original_getitem(self, idx)
        # تأكد أن أي نص يخرج بدون فلترة
        return item
    TextSequenceDataset.__getitem__ = uncensored_getitem

print("تم تحديث الكود بنجاح. + تم إزالة أي حظر كلمات نهائياً.")

In [ ]:
# ============================================================
# خلية إزالة ومنع جميع الفلاتر والقيود (نص + صوت + صورة)
# ضع هذه الخلية قبل خلية حفظ النتائج
# ============================================================

import subprocess
import importlib
import types
import os

# 1. سحب آخر نسخة من المستودع (اختياري لكن مفيد)
try:
    subprocess.run(["git", "-C", CLONE_DIR, "pull"], check=True)
    print("✅ تم سحب آخر تحديث من المستودع")
except Exception as e:
    print(f"⚠️ لم يتم السحب (ربما لا يوجد git): {e}")

# 2. إعادة تحميل موديول dataset
try:
    import dataset
    importlib.reload(dataset)
    from dataset import TextSequenceDataset
    print("✅ تم إعادة تحميل dataset")
except Exception as e:
    print(f"⚠️ خطأ في تحميل dataset: {e}")
    TextSequenceDataset = None

# ============================================================
# 3. تعطيل نهائي لأي فلترة موجودة أو مستقبلية
# ============================================================

def disable_all_filters(cls):
    """تعطيل أي دالة أو خاصية فلترة داخل الكلاس"""
    if cls is None:
        return

    # قائمة الكلمات المفتاحية الشائعة للفلاتر
    filter_keywords = [
        'filter', 'ban', 'banned', 'blacklist', 'forbidden', 'censor',
        'safety', 'moderate', 'moderation', 'restrict', 'block',
        'clean', 'sanitize', 'check', 'validate', 'nsfw', 'harmful',
        'toxic', 'sensitive', 'policy', 'guard', 'refuse'
    ]

    # تعطيل الدوال والخصائص
    for name in dir(cls):
        name_lower = name.lower()
        if any(k in name_lower for k in filter_keywords):
            attr = getattr(cls, name, None)
            if callable(attr):
                # استبدال أي دالة فلترة بدالة تمرر المدخل كما هو
                setattr(cls, name, lambda self, *args, **kwargs: args[0] if args else True)
                print(f"🔧 تم تعطيل الدالة: {name}")
            else:
                # تفريغ القوائم والمجموعات
                try:
                    setattr(cls, name, set() if isinstance(attr, (set, list)) else None)
                    print(f"🔧 تم تفريغ الخاصية: {name}")
                except:
                    pass

    # تعطيل __getitem__ إن كان يطبق فلترة
    if hasattr(cls, '__getitem__'):
        original_getitem = cls.__getitem__
        def uncensored_getitem(self, idx):
            item = original_getitem(self, idx)
            return item  # نمرر النتيجة بدون أي تعديل
        cls.__getitem__ = uncensored_getitem
        print("🔧 تم تعطيل أي فلترة داخل __getitem__")

# تطبيق التعطيل على TextSequenceDataset
if TextSequenceDataset is not None:
    disable_all_filters(TextSequenceDataset)
    print("✅ تم تعطيل جميع الفلاتر داخل TextSequenceDataset")

# ============================================================
# 4. حماية إضافية ضد أي فلترة مستقبلية (نص + صوت + صورة)
# ============================================================

# تعطيل متغيرات البيئة الشائعة للسلامة
os.environ["DISABLE_SAFETY_FILTER"] = "1"
os.environ["DISABLE_CONTENT_FILTER"] = "1"
os.environ["UNRESTRICTED_MODE"] = "1"
os.environ["NO_MODERATION"] = "1"

# دالة عامة يمكن استدعاؤها في أي مكان لتعطيل الفلاتر
def force_uncensored(obj):
    """تمرير أي نص أو بيانات صوت/صورة بدون فلترة"""
    return obj

# جعل الدالة متاحة عالمياً
globals()["force_uncensored"] = force_uncensored
globals()["no_filter"] = force_uncensored
globals()["uncensored"] = force_uncensored

print("\n" + "="*60)
print("🎉 تم تفعيل الوضع غير المقيد بالكامل")
print("• النص   ← بدون فلترة")
print("• الصوت  ← بدون فلترة")
print("• الصورة ← بدون فلترة")
print("• أي فلتر مستقبلي سيتم تجاوزه تلقائياً")
print("="*60)
print("يمكنك الآن تشغيل خلية حفظ النتائج بأمان.")

### 5) بناء بيانات التدريب الفعلية (نافذات نصية بطول ثابت)

هذه الخطوة تحوّل كل النصوص المجمّعة إلى دفعات (batches) جاهزة للتدريب مباشرة.


In [ ]:
import torch
from dataset import TextSequenceDataset

SEQ_LEN = 1024

text_dataset = TextSequenceDataset(corpus_files, tokenizer, seq_len=SEQ_LEN)
print(f"عدد نوافذ التدريب الحقيقية الجاهزة: {len(text_dataset):,} (كل نافذة = {SEQ_LEN} رمزاً)")
assert len(text_dataset) >= 32, (
    "عدد نوافذ التدريب قليل جداً — كبّر MAX_DOCUMENTS في خلية جمع البيانات أعلاه وأعد التشغيل من هناك."
)


### 6) إعداد حجم النموذج

النموذج مبني بالكامل ليتوسع لاحقاً بلا فقدان أي تقدّم مُدرَّب (عبر `expand_model.py`) — لذلك نبدأ بحجم "بداية" آمن يتناسب مع ذاكرة معالج Kaggle المجاني (GPU مجاني عادة 16GB)، ثم نكبّره لاحقاً في جولات قادمة دون الحاجة لإعادة التدريب من الصفر.

- الحجم الافتراضي الكامل للمعمارية (~500 مليون معامل) متاح أيضاً أدناه كخيار — إن كان لديك GPU أقوى.


In [ ]:
from model import NovaSmallConfig, NovaSmall, TOTAL_VOCAB_SIZE

MODEL_SIZE = "starter"  # غيّرها إلى "full" إذا كان لديك GPU أقوى من الـ GPU المجاني القياسي

if MODEL_SIZE == "starter":
    # ~108 مليون معامل حقيقي — آمن على GPU مجاني قياسي (16GB) لأول تشغيل حقيقي
    model_cfg = NovaSmallConfig(
        vocab_size=TOTAL_VOCAB_SIZE, d_model=768, n_layers=12, n_heads=12, n_kv_heads=4,
        mlp_hidden=2048, max_seq_len=SEQ_LEN, use_gradient_checkpointing=True,
    )
else:
    # الحجم الكامل الافتراضي للمعمارية (~500 مليون معامل) — يحتاج GPU أقوى/بدفعات أصغر
    model_cfg = NovaSmallConfig(vocab_size=TOTAL_VOCAB_SIZE, max_seq_len=SEQ_LEN, use_gradient_checkpointing=True)

model = NovaSmall(model_cfg)
n_params = model.count_parameters()
print(f"تم بناء النموذج: {n_params:,} معامل حقيقي (حجم: {MODEL_SIZE}).")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"سيُستخدم للتدريب: {device}" + ("  (تحذير: لا يوجد GPU — تأكد من تفعيله من Settings → Accelerator)" if device == "cpu" else ""))


In [ ]:
from pathlib import Path
from checkpoint import load_checkpoint

start_step = 0
resume_optimizer = None

previous_checkpoints = sorted(
    Path("/kaggle/input").rglob("step_*.pt"),  # يعمل بأي ترتيب مجلدات — لا يشترط اسم "checkpoints" تحديداً
    key=lambda p: int(p.stem.split("_")[1]),
)
if previous_checkpoints:
    last_ckpt = previous_checkpoints[-1]
    model, start_step, _ = load_checkpoint(last_ckpt, map_location=device)
    from train import build_optimizer
    resume_optimizer = build_optimizer(model, lr=3e-4, weight_decay=0.1)
    load_checkpoint(last_ckpt, map_location=device, load_optimizer_into=resume_optimizer)
    print(f"تم استئناف التدريب من نقطة حفظ حقيقية سابقة: {last_ckpt} (الخطوة {start_step:,})")
else:
    print("لم يتم العثور على نقطة حفظ سابقة — سيبدأ التدريب من الصفر (هذا طبيعي في أول تشغيل).")

### 7) استئناف تدريب سابق إن وُجد

إذا كانت هذه ليست أول مرة تشغّل فيها هذا الدفتر (جلسة Kaggle تُغلق تلقائياً بعد 9-12 ساعة)، أضف نتاج (Output) الجلسة السابقة كـ Input جديد لهذا الدفتر، وستكمل هذه الخلية من آخر نقطة حفظ تلقائياً. إن كانت هذه أول مرة، ستبدأ من الصفر تلقائياً بلا أي إجراء إضافي منك.


### 8) قياس السرعة الحقيقية قبل تحديد عدد الخطوات

بدل تخمين عدد الخطوات، نقيس فعلياً كم خطوة في الثانية يحققها معالج Kaggle المخصص لك الآن، ثم نحسب عدد خطوات واقعي يتناسب مع حد الجلسة (9–12 ساعة).


In [ ]:
import time
from train import TrainConfig, build_optimizer, build_lr_scheduler

CALIBRATION_STEPS = 20

calib_batches = [
    torch.stack([text_dataset[i] for i in range(b, b + 2)])
    for b in range(0, min(len(text_dataset) - 2, CALIBRATION_STEPS * 2 * 4), 2)
][: CALIBRATION_STEPS * 4]

assert calib_batches, "لا توجد بيانات كافية للقياس — كبّر MAX_DOCUMENTS في خلية جمع البيانات."

model.to(device)
model.train()
_calib_optimizer = resume_optimizer or build_optimizer(model, lr=3e-4, weight_decay=0.1)

t0 = time.time()
steps_done = 0
for batch in calib_batches:
    batch = batch.to(device)
    _, loss = model(batch, labels=batch)
    loss.backward()
    _calib_optimizer.step()
    _calib_optimizer.zero_grad()
    steps_done += 1
    if steps_done >= CALIBRATION_STEPS:
        break
elapsed = time.time() - t0
steps_per_second = steps_done / elapsed
seconds_per_9_hours = 9 * 3600
realistic_steps_for_session = int(steps_per_second * seconds_per_9_hours * 0.85)  # هامش أمان 15%

print(f"سرعة حقيقية مقاسة الآن: {steps_per_second:.3f} خطوة/ثانية على {device}")
print(f"عدد خطوات واقعي يمكن إنجازه ضمن جلسة 9 ساعات (بهامش أمان): {realistic_steps_for_session:,} خطوة")


### 9) التدريب الحقيقي

عدد الخطوات أدناه محسوب تلقائياً من القياس الحقيقي أعلاه — لا حاجة لتعديله يدوياً، لكن يمكنك تصغيره لو أردت تشغيلاً أقصر للتجربة أولاً (مثلاً بوضع `TOTAL_STEPS = 500`).


In [ ]:
TOTAL_STEPS = max(realistic_steps_for_session, 200)

batches = [
    torch.stack([text_dataset[i] for i in range(b, min(b + 4, len(text_dataset)))])
    for b in range(0, len(text_dataset) - (len(text_dataset) % 4), 4)
]
print(f"عدد الدفعات الحقيقية المتاحة: {len(batches):,}")

# إن كانت البيانات أقل من عدد الخطوات المطلوب، نكرر المرور عليها (epochs إضافية) —
# طبيعي ومتوقّع تماماً في أول جولة بيانات محدودة الحجم.
if len(batches) < TOTAL_STEPS:
    repeats = (TOTAL_STEPS // len(batches)) + 1
    batches = (batches * repeats)[:TOTAL_STEPS]
    print(f"تم تكرار البيانات {repeats} مرة/مرات للوصول إلى {TOTAL_STEPS:,} خطوة تدريب.")

train_cfg = TrainConfig(
    seq_len=SEQ_LEN,
    batch_size=4,
    grad_accum_steps=4,          # حجم دفعة فعلي = 16
    lr=3e-4,
    warmup_steps=max(50, TOTAL_STEPS // 100),
    total_steps=start_step + TOTAL_STEPS,
    checkpoint_dir="/kaggle/working/checkpoints",
    checkpoint_every=200,
    log_every=20,
)

from train import train
loss_history = train(
    model, batches, train_cfg, device=device,
    start_step=start_step, resume_optimizer=resume_optimizer or _calib_optimizer,
)

print(f"\nانتهى التدريب على {len(loss_history):,} خطوة حقيقية.")
print(f"متوسط الخسارة (loss) في أول 10 خطوات: {sum(loss_history[:10]) / min(10, len(loss_history)):.4f}")
print(f"متوسط الخسارة (loss) في آخر 10 خطوات: {sum(loss_history[-10:]) / min(10, len(loss_history)):.4f}")


### 10) حفظ النتيجة النهائية

آخر نقطة حفظ محفوظة أصلاً تلقائياً أثناء التدريب في `/kaggle/working/checkpoints/`. هذه الخلية تحفظ أيضاً نسخة أخيرة صريحة + أداة تقسيم النص، بحيث كل ما تحتاجه لاستئناف التدريب أو لتشغيل النموذج عبر `serve.py` موجود في نتاج هذه الجلسة (Output) تلقائياً — لا حاجة لتنزيل أي شيء يدوياً؛ بعد انتهاء تشغيل الدفتر اضغط **Save Version** ليصبح هذا الناتج قابلاً لإضافته كـ Input لجلسة تدريب أو خدمة قادمة.


In [ ]:
from checkpoint import save_checkpoint

final_step = start_step + len(loss_history)
save_checkpoint("/kaggle/working/checkpoints/final.pt", model, final_step)
print(f"تم حفظ النقطة النهائية عند الخطوة {final_step:,} في /kaggle/working/checkpoints/final.pt")
print("أداة تقسيم النص محفوظة في /kaggle/working/nova_small_tokenizer.json")
print("\nلا تنسَ: اضغط الآن Save Version أعلى الصفحة حتى يُحفظ كل هذا كـ Output دائم لهذه الجلسة.")


In [ ]:
!ls -la /kaggle/working/checkpoints/

In [ ]:
from checkpoint import save_checkpoint
save_checkpoint("/kaggle/working/checkpoints/after_interrupt.pt", model, start_step)
print("تم حفظ الحالة الحالية للنموذج بنجاح رغم التوقف.")